# Module 3 — Class 2: Data Encoding and Transformation

**Lecture reference:** `MODULE_3_OLIST_REWRITE.md` § Class 3-2

**Today:** turn categorical text columns (`payment_type`, `customer_state`) into numbers, and reshape skewed numeric columns (`freight_value`) so linear models can fit them.

## Why this matters today

Models can't read text. `payment_type='credit_card'` is useless to logistic regression until we convert it. Similarly, `freight_value` ranges 0 → 400+ reais with a heavy right tail; log scale lifts the shape into something a linear model can fit.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder

orders = pd.read_parquet('orders_step1.parquet')
items = pd.read_csv('olist_data/olist_order_items_dataset.csv')
payments = pd.read_csv('olist_data/olist_order_payments_dataset.csv')
print('orders:', orders.shape)
print('items :', items.shape)
print('pay   :', payments.shape)

## Section 1 — Aggregate items + payments to one row per order

In [ ]:
# One order can have multiple line items — collapse to per-order totals
order_totals = items.groupby('order_id').agg(
    num_items=('order_item_id', 'count'),
    total_price=('price', 'sum'),
    total_freight=('freight_value', 'sum'),
).reset_index()

# Primary payment method per order = the one with the largest payment_value
primary_pay = (payments
    .sort_values('payment_value', ascending=False)
    .drop_duplicates('order_id')[['order_id', 'payment_type', 'payment_installments']])

df = orders.merge(order_totals, on='order_id', how='left') \
           .merge(primary_pay,  on='order_id', how='left')
print(df.shape)
df.head(3)

## Section 2 — One-hot encoding for payment_type (low cardinality)

In [ ]:
print(df['payment_type'].value_counts(dropna=False))

# Only 4-5 categories → one-hot is fine
pay_dummies = pd.get_dummies(df['payment_type'], prefix='pay', dummy_na=True)
print(pay_dummies.head())
df = pd.concat([df, pay_dummies], axis=1)

## Section 3 — Log-transform the right-skewed `total_freight`

Most freight is 5-30 reais. A long tail goes to 400+. Linear models struggle with that shape. `np.log1p(x) = log(1+x)` handles `x=0` and pulls the tail in.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
df['total_freight'].dropna().hist(bins=50, ax=axes[0])
axes[0].set_title('Original (heavy tail)')
df['total_freight'].dropna().pipe(np.log1p).hist(bins=50, ax=axes[1])
axes[1].set_title('After log1p (closer to bell)')
plt.tight_layout(); plt.show()

In [ ]:
df['log_freight'] = np.log1p(df['total_freight'].fillna(0))

## Section 4 — Standardise the numeric columns (mean 0, std 1)

**Important:** for now we just demonstrate. In Class 3-5 we'll wrap this in a Pipeline that fits ONLY on training data (no leakage).

In [ ]:
scaler = StandardScaler()
num_cols = ['log_freight', 'total_price', 'num_items']
df[num_cols + ['log_freight_scaled', 'total_price_scaled', 'num_items_scaled']]
df[[c + '_scaled' for c in num_cols]] = scaler.fit_transform(df[num_cols].fillna(0))
df[[c + '_scaled' for c in num_cols]].describe().round(2)

## Section 5 — Save Stage-2 output

In [ ]:
df.to_parquet('orders_step2.parquet', index=False)
print('orders_step2.parquet:', df.shape)

## ✅ Quick Check

1. Why one-hot encode `payment_type` but use target-encoding (later) for `customer_state` if it had 27 levels?  
2. Why does `log1p(freight)` help linear regression but make no difference to a Random Forest?  
3. What's the leakage trap when standardising numeric columns?

## 📝 Homework (Bronze)

1. Run every cell. Save `orders_step2.parquet`.  
2. Plot the histogram of `total_price` before and after log-transform.  
3. One-hot encode `customer_state`. How many new columns?

## 🔥 Homework (Gold)

1. Implement target encoding for `customer_state` (mean of `is_late` per state, computed on TRAINING split only — research how `category_encoders.TargetEncoder` works).  
2. Compare its memory footprint with one-hot encoding.  
3. Half-page rationale: which encoding choice for state on Olist, and why?